# 01 — Dataset Inspection

Verifies the RADS dataset pulled from Roboflow: class distribution, image counts per split, sample images per class. Run this BEFORE training so any silent class-order bug fails loudly here, not 8 hours into a training run.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
import yaml
from collections import Counter

from src.data.roboflow_pull import pull
from src.paths import CLASS_NAMES, RADS_DATA

dataset = pull()
print('Dataset root:', dataset)
cfg = yaml.safe_load((dataset/'data.yaml').read_text())
print('Classes:', cfg['names'])
assert list(cfg['names'].values()) == CLASS_NAMES if isinstance(cfg['names'], dict) else cfg['names'] == CLASS_NAMES

In [ ]:
# Image counts per split
for split in ['train', 'valid', 'test']:
    n = len(list((dataset/split/'images').glob('*.*')))
    print(f'{split:8s} {n:6d} images')

In [ ]:
# Class-instance count per split (counts boxes, not images)
import pandas as pd
rows = []
for split in ['train', 'valid', 'test']:
    counts = Counter()
    for lbl in (dataset/split/'labels').glob('*.txt'):
        for line in lbl.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    for cls_id, n in counts.items():
        rows.append({'split': split, 'class': CLASS_NAMES[cls_id], 'instances': n})
df = pd.DataFrame(rows).pivot(index='class', columns='split', values='instances').fillna(0).astype(int)
df

In [ ]:
# Sample 3 random images per class
import random, cv2, matplotlib.pyplot as plt
random.seed(42)
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ci, cname in enumerate(CLASS_NAMES):
    matching = []
    for lbl in (dataset/'train'/'labels').glob('*.txt'):
        ids = [int(l.split()[0]) for l in lbl.read_text().splitlines() if l.strip()]
        if ci in ids:
            matching.append(lbl)
    picks = random.sample(matching, min(3, len(matching)))
    for ai, lbl in enumerate(picks):
        img_path = next((dataset/'train'/'images').glob(lbl.stem + '.*'))
        im = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        axes[ci, ai].imshow(im); axes[ci, ai].set_title(cname); axes[ci, ai].axis('off')
plt.tight_layout(); plt.show()